In [4]:
import os
import pandas as pd
from datetime import datetime


SOURCE_CSV   = "/home/jovyan/work/alterBigDateProj/bank_transactions_data_2_augmented_clean_2.csv"
LANDING_ZONE = "/home/jovyan/work/alterBigDateProj/data_raw_bank_pings/"

os.makedirs(LANDING_ZONE, exist_ok=True)

BATCH_SIZE = 50

# SIMULATOR — Run ONCE only to pre-generate all batches
# All columns are kept here — dropping happens in transformation
# Airflow tracks batches by batch_file (full path)

def run_simulator():
    print("\n" + "="*60)
    print("  bank transaction simulator:")
    print("  Splitting CSV into batches ==> landing zone")
    print("="*60 + "\n")

    try:
        df = pd.read_csv(SOURCE_CSV)
        df.columns = df.columns.str.strip()

        total_rows    = len(df)
        total_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

        print(f"  CSV Loaded Successfully")
        print(f"  Total rows    : {total_rows:,}")
        print(f"  Total columns : {len(df.columns)}")
        print(f"  Columns       : {list(df.columns)}")
        print(f"  Batch size    : {BATCH_SIZE} rows")
        print(f"  Total batches : {total_batches:,}")
        print(f"  Landing zone  : {LANDING_ZONE}\n")
        print("-"*60)

    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    for batch_num in range(1, total_batches + 1):
        start = (batch_num - 1) * BATCH_SIZE
        end   = start + BATCH_SIZE

        chunk = df.iloc[start:end].copy()

        # Metadata columns 
        chunk["source"]      = "bank_transactions"
        chunk["ingested_at"] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        chunk["batch_id"]    = f"BANK-BATCH-{batch_num:04d}"

        # ── Save directly to landing zone (no subfolders) ──
        file_name = f"bank_batch_{batch_num:04d}.json"
        out_path  = os.path.join(LANDING_ZONE, file_name)
        chunk.to_json(out_path, orient='records', lines=True)

        print(
            f"  Batch {batch_num:>5} / {total_batches} | "
            f"Rows: {start}–{min(end, total_rows) - 1:<6} | "
            f"Records: {len(chunk):<3} | "
            f"File: {file_name}|Time: {chunk['ingested_at'].iloc[0]} "
        )

    print(f"\n{'='*60}")
    print(f"  SIMULATION COMPLETE ")
    print(f"  {total_batches:,} batch files saved in landing zone")
    print(f"  Next step → Trigger Airflow DAG: bank_etl_pipeline")
    print(f"{'='*60}\n")



if __name__ == "__main__":
    try:
        run_simulator()
    except KeyboardInterrupt:
        print("\n  Simulator stopped by user.")


  bank transaction simulator:
  Splitting CSV into batches ==> landing zone

  CSV Loaded Successfully
  Total rows    : 50,000
  Total columns : 15
  Columns       : ['TransactionID', 'AccountID', 'TransactionAmount', 'TransactionDate', 'TransactionType', 'Location', 'DeviceID', 'IP Address', 'MerchantID', 'Channel', 'CustomerAge', 'CustomerOccupation', 'TransactionDuration', 'LoginAttempts', 'AccountBalance']
  Batch size    : 50 rows
  Total batches : 1,000
  Landing zone  : /home/jovyan/work/alterBigDateProj/data_raw_bank_pings/

------------------------------------------------------------
  Batch     1 / 1000 | Rows: 0–49     | Records: 50  | File: bank_batch_0001.json|Time: 2026-05-05 01:38:55 
  Batch     2 / 1000 | Rows: 50–99     | Records: 50  | File: bank_batch_0002.json|Time: 2026-05-05 01:38:55 
  Batch     3 / 1000 | Rows: 100–149    | Records: 50  | File: bank_batch_0003.json|Time: 2026-05-05 01:38:55 
  Batch     4 / 1000 | Rows: 150–199    | Records: 50  | File: bank_